In [1]:
import os
import random
import pandas as pd
import numpy as np
import tensorflow as tf
from PIL import Image

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


I0000 00:00:1776288985.792669   67347 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
CLASS_NAMES = [
    "Black Sea Sprat",      # 0
    "Gilt-Head Bream",      # 1
    "Hourse Mackerel",      # 2
    "Red Mullet",           # 3
    "Red Sea Bream",        # 4
    "Sea Bass",             # 5
    "Shrimp",               # 6
    "Striped Red Mullet",   # 7
    "Trout",                # 8
]
print(CLASS_NAMES)

# set to the folder that directly contains the 9 label folders.
src_dir = '/home/groggy/data/a-large-scale-fish-dataset/2/Fish_Dataset/Fish_Dataset'
# set to a NEW output folder where the notebook will copy images,
# removing any subfolders whose name ends with "GT".
clean_dir = '/home/groggy/data/a-large-scale-fish-dataset/2/Fish_Dataset/Fish_Dataset_clean'
# After cleaning, clean_dir should look like:
# <clean_dir>/<Label_i>/<image_files...>

SEED = 42
random.seed(SEED)

['Black Sea Sprat', 'Gilt-Head Bream', 'Hourse Mackerel', 'Red Mullet', 'Red Sea Bream', 'Sea Bass', 'Shrimp', 'Striped Red Mullet', 'Trout']


In [3]:
selected_rows = []

for cls in CLASS_NAMES:
    class_dir = os.path.join(src_dir, cls)
    if not os.path.isdir(class_dir):
        raise FileNotFoundError(f"Missing class folder: {class_dir}")

    # The dataset has an extra nested folder level; we collect from the subfolder(s) that match the class
    # and skip anything ending with GT.
    candidate_img_paths = []
    for subfolder in os.listdir(class_dir):
        sub_path = os.path.join(class_dir, subfolder)
        if not os.path.isdir(sub_path):
            continue
        if subfolder.endswith("GT"):
            continue
        if subfolder != cls:
            continue

        for fname in os.listdir(sub_path):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                candidate_img_paths.append(os.path.join(sub_path, fname))

    if len(candidate_img_paths) < 3:
        raise ValueError(f"Not enough images for class '{cls}'. Found {len(candidate_img_paths)}")

    chosen = random.sample(candidate_img_paths, 3)
    for p in chosen:
        selected_rows.append({"img_path": p, "true_label_name": cls})

mini_eval_df = pd.DataFrame(selected_rows)
print("Mini eval size:", len(mini_eval_df))
mini_eval_df.head()

Mini eval size: 27


,img_path,true_label_name
0,/home/groggy/data/a-large-scale-fish-dataset/2...,Black Sea Sprat
1,/home/groggy/data/a-large-scale-fish-dataset/2...,Black Sea Sprat
2,/home/groggy/data/a-large-scale-fish-dataset/2...,Black Sea Sprat
3,/home/groggy/data/a-large-scale-fish-dataset/2...,Gilt-Head Bream
4,/home/groggy/data/a-large-scale-fish-dataset/2...,Gilt-Head Bream


In [11]:
model = tf.keras.models.load_model("Fish_MN.keras")
model2 = tf.keras.models.load_model("Models/MobileNetV3Large_Improved.keras")
# Build mapping index -> label name (based on output order)
# IMPORTANT: this assumes the model output index ordering matches CLASS_NAMES order.
idx_to_name = {i: name for i, name in enumerate(CLASS_NAMES)}

def load_and_preprocess_image(path, target_size=(224, 224)):
    img = Image.open(path).convert("RGB").resize(target_size)
    arr = np.array(img).astype("float32")
    #arr = preprocess_input(arr)  # MobileNetV2 specific
    return arr

# Predict
pred_records = []

for i, row in mini_eval_df.iterrows():
    x = load_and_preprocess_image(row["img_path"])
    x = np.expand_dims(x, axis=0)  # batch dimension

    probs = model2.predict(x, verbose=0)[0]  # shape: (num_classes,)
    pred_idx = int(np.argmax(probs))
    pred_name = idx_to_name[pred_idx]
    true_name = row["true_label_name"]

    pred_records.append({
        "img_path": row["img_path"],
        "true_label_name": true_name,
        "pred_label_name": pred_name,
        "pred_prob": float(probs[pred_idx]),
        "pred_idx": pred_idx
    })

pred_df = pd.DataFrame(pred_records)

pred_df.sort_values("true_label_name").reset_index(drop=True).head(30)

,img_path,true_label_name,pred_label_name,pred_prob,pred_idx
0,/home/groggy/data/a-large-scale-fish-dataset/2...,Black Sea Sprat,Black Sea Sprat,1.000000,0
1,/home/groggy/data/a-large-scale-fish-dataset/2...,Black Sea Sprat,Black Sea Sprat,1.000000,0
2,/home/groggy/data/a-large-scale-fish-dataset/2...,Black Sea Sprat,Black Sea Sprat,1.000000,0
3,/home/groggy/data/a-large-scale-fish-dataset/2...,Gilt-Head Bream,Gilt-Head Bream,0.999878,1
4,/home/groggy/data/a-large-scale-fish-dataset/2...,Gilt-Head Bream,Gilt-Head Bream,0.999999,1
5,/home/groggy/data/a-large-scale-fish-dataset/2...,Gilt-Head Bream,Gilt-Head Bream,0.999994,1
6,/home/groggy/data/a-large-scale-fish-dataset/2...,Hourse Mackerel,Hourse Mackerel,1.000000,2
7,/home/groggy/data/a-large-scale-fish-dataset/2...,Hourse Mackerel,Hourse Mackerel,1.000000,2
8,/home/groggy/data/a-large-scale-fish-dataset/2...,Hourse Mackerel,Hourse Mackerel,1.000000,2
9,/home/groggy/data/a-large-scale-fish-dataset/2...,Red Mullet,Red Mullet,1.000000,3


In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         4,617 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,239,453 (16.17 MB)

 Trainable params: 660,489 (2.52 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 1,320,980 (5.04 MB)